In [1]:
"""
==============================================================================
 PIPELINE BASELINE — XGBOOST + ESTACIÓN (opcional)
==============================================================================
 Versión XGBoost del baseline, con la MISMA estructura modular que la de
 Random Forest para permitir comparación directa entre ambos modelos.

 Requiere: pip install xgboost

 BLOQUES:
   1 — Configuración        2 — Carga + (one-hot estación)
   3 — Validación           4 — Entrenamiento/evaluación
   5 — Reporte              6 — Main
==============================================================================
"""

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Total_Parasitos_",       # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"   : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],

    # --- Estación ---
    "usar_estacion" : True,          # True = agrega Estacion con one-hot
    "col_estacion"  : "Estacion",

    # --- Validación ---
    "validacion" : "aleatoria",      # "temporal" o "aleatoria"
    "n_folds"    : 5,
    "random_state": 42,

    # --- Hiperparámetros del modelo ---
    # Baseline limpio: solo lo esencial, el resto por defecto de XGBoost.
    "xgb_params" : {
        "random_state": 42,
        "n_jobs": -1,
        "objective": "reg:squarederror",   # regresión estándar
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN (con one-hot de estación)
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])

    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])

    if config["usar_estacion"]:
        col = config["col_estacion"]
        dummies = pd.get_dummies(df[col], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()

    return X, y, df, features


# ============================================================================
# BLOQUE 3 — VALIDACIÓN
# ============================================================================
def obtener_folds(config, X):
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
    elif config["validacion"] == "aleatoria":
        splitter = KFold(n_splits=config["n_folds"], shuffle=True,
                         random_state=config["random_state"])
    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")
    return list(splitter.split(X))


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    resultados = []
    for i, (train_idx, test_idx) in enumerate(folds):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        modelo = XGBRegressor(**config["xgb_params"])
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        resultados.append({
            "fold"   : i + 1,
            "n_train": len(train_idx),
            "n_test" : len(test_idx),
            "RMSE"   : np.sqrt(mean_squared_error(y_test, y_pred)),
            "MAE"    : mean_absolute_error(y_test, y_pred),
            "R2"     : r2_score(y_test, y_pred),
        })
    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE
# ============================================================================
def reportar(config, resultados, features):
    print("=" * 70)
    print("  BASELINE — XGBOOST + ESTACIÓN")
    print("=" * 70)
    print(f"  Target     : {config['target']}")
    print(f"  Estación   : {'Sí (one-hot)' if config['usar_estacion'] else 'No'}")
    print(f"  Features   : {len(features)}  →  {features}")
    print(f"  Validación : {config['validacion']}  ({config['n_folds']} folds)")
    print(f"  Modelo     : XGBRegressor (parámetros por defecto)")

    print("\n" + "-" * 70)
    print(f"  {'Fold':<6}{'n_train':<10}{'n_test':<9}{'RMSE':<10}{'MAE':<10}{'R²':<10}")
    print("-" * 70)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_train']:<10}{r['n_test']:<9}"
              f"{r['RMSE']:<10.4f}{r['MAE']:<10.4f}{r['R2']:<10.4f}")

    rmse = np.array([r["RMSE"] for r in resultados])
    mae  = np.array([r["MAE"]  for r in resultados])
    r2   = np.array([r["R2"]   for r in resultados])
    print("-" * 70)
    print(f"  {'MEDIA':<25}{rmse.mean():<10.4f}{mae.mean():<10.4f}{r2.mean():<10.4f}")
    print(f"  {'DESV. EST.':<25}{rmse.std():<10.4f}{mae.std():<10.4f}{r2.std():<10.4f}")
    print("=" * 70)


# ============================================================================
# BLOQUE 6 — MAIN
# ============================================================================
def main():
    X, y, df, features = cargar_datos(CONFIG)
    folds = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, features)


if __name__ == "__main__":
    main()

  BASELINE — XGBOOST + ESTACIÓN
  Target     : Total_Parasitos_
  Estación   : Sí (one-hot)
  Features   : 9  →  ['Temperatura_C', 'Salinidad_psu', 'Capa_Mezcla_m', 'Altura_Mar_m', 'Velocidad_Corriente_m_s', 'Est_Invierno', 'Est_Otoño', 'Est_Primavera', 'Est_Verano']
  Validación : aleatoria  (5 folds)
  Modelo     : XGBRegressor (parámetros por defecto)

----------------------------------------------------------------------
  Fold  n_train   n_test   RMSE      MAE       R²        
----------------------------------------------------------------------
  1     1902      476      1.9029    1.2211    0.7498    
  2     1902      476      2.2302    1.3555    0.6994    
  3     1902      476      1.8941    1.2129    0.7196    
  4     1903      475      1.9120    1.2495    0.6841    
  5     1903      475      2.0520    1.3465    0.6400    
----------------------------------------------------------------------
  MEDIA                    1.9982    1.2771    0.6986    
  DESV. EST.           

In [2]:
"""
==============================================================================
 XGBOOST — Baseline con registro de ejecuciones en Excel
==============================================================================
 Misma estructura modular que la versión Random Forest, para comparación
 directa. Cada ejecución se AÑADE a un Excel acumulativo propio de XGBoost.

 Requiere: pip install xgboost

 Hojas del Excel:
   - Ejecuciones    : historial; una fila por corrida (config + métricas)
   - Detalle_Folds  : desglose por fold de la última corrida
   - Importancia    : (preparada) significancia de cada variable
   - SHAP           : (preparada) resultados de SHAP value

 Validaciones disponibles: "aleatoria", "temporal", "holdout".

 BLOQUES:
   1 Configuración  2 Carga/preparación  3 Validación
   4 Entrenamiento/evaluación  5 Reporte consola  6 Guardado Excel  7 Main
==============================================================================
"""

import os
from datetime import datetime
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             median_absolute_error, r2_score)
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# Todo lo editable para cada experimento vive aquí.
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_XGBoost.xlsx",    # bitácora propia de XGBoost
    "target"       : "Total_Parasitos_",     # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],
    "usar_estacion": True,
    "col_estacion" : "Estacion",

    # Validación: "aleatoria" | "temporal" | "holdout"
    "validacion"   : "aleatoria",
    "n_folds"      : 5,          # aplica a "aleatoria" y "temporal"
    "test_size"    : 0.20,       # aplica solo a "holdout" (último 20% temporal)
    "random_state" : 42,

    "xgb_params"   : {
        "random_state": 42,
        "n_jobs": -1,
        "objective": "reg:squarederror",
    },
    "nota"         : "",         # texto libre para identificar la corrida
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# Filtra registros con target, ordena por fecha y arma la matriz de features
# (con one-hot de estación si está activada).
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — VALIDACIÓN
# Devuelve los pares (train, test) según la estrategia:
#   aleatoria → KFold con shuffle
#   temporal  → TimeSeriesSplit (varios folds, respeta el tiempo)
#   holdout   → un solo corte temporal final (último test_size va a prueba)
# ============================================================================
def obtener_folds(config, X):
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
        return list(splitter.split(X))
    elif config["validacion"] == "aleatoria":
        splitter = KFold(n_splits=config["n_folds"], shuffle=True,
                         random_state=config["random_state"])
        return list(splitter.split(X))
    elif config["validacion"] == "holdout":
        n = len(X)
        corte = int(n * (1 - config["test_size"]))
        return [(np.arange(0, corte), np.arange(corte, n))]
    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# Entrena un XGBoost por fold y calcula las cinco métricas de regresión.
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    resultados = []
    for i, (tr, te) in enumerate(folds):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        modelo = XGBRegressor(**config["xgb_params"])
        modelo.fit(Xtr, ytr)
        pred = modelo.predict(Xte)

        rmse  = np.sqrt(mean_squared_error(yte, pred))
        media = yte.mean()
        resultados.append({
            "fold": i + 1, "n_train": len(tr), "n_test": len(te),
            "RMSE": rmse,
            "MAE": mean_absolute_error(yte, pred),
            "R2": r2_score(yte, pred),
            "MedAE": median_absolute_error(yte, pred),
            "RMSE_norm": rmse / media if media != 0 else np.nan,
        })
    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE POR CONSOLA
# Imprime la tabla por fold y los promedios (lectura rápida en terminal).
# ============================================================================
def reportar(config, resultados, features):
    print("=" * 86)
    print("  XGBOOST —", config["target"],
          "| estación:", "sí" if config["usar_estacion"] else "no",
          "| validación:", config["validacion"])
    print("=" * 86)
    print(f"  {'Fold':<6}{'n_test':<8}{'RMSE':<10}{'MAE':<10}{'R²':<10}"
          f"{'MedAE':<10}{'RMSE_norm':<12}")
    print("-" * 86)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_test']:<8}{r['RMSE']:<10.4f}{r['MAE']:<10.4f}"
              f"{r['R2']:<10.4f}{r['MedAE']:<10.4f}{r['RMSE_norm']:<12.4f}")
    if len(resultados) > 1:
        col = lambda k: np.array([r[k] for r in resultados])
        print("-" * 86)
        print(f"  {'MEDIA':<14}{col('RMSE').mean():<10.4f}{col('MAE').mean():<10.4f}"
              f"{col('R2').mean():<10.4f}{col('MedAE').mean():<10.4f}"
              f"{col('RMSE_norm').mean():<12.4f}")
    print("=" * 86)


# ============================================================================
# BLOQUE 6 — GUARDADO EN EXCEL (acumulativo)
# Si el Excel no existe, lo crea con sus hojas. Si existe, AÑADE la corrida
# a "Ejecuciones" y reemplaza "Detalle_Folds" con el desglose más reciente.
# Las hojas "Importancia" y "SHAP" se dejan preparadas para llenarse luego.
# ============================================================================
def guardar_excel(config, resultados, features):
    ruta = config["excel_salida"]
    col = lambda k: np.array([r[k] for r in resultados])

    # En holdout hay un solo fold: el promedio es ese mismo valor.
    fila = {
        "Fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "Modelo": "XGBoost",
        "Target": config["target"],
        "Estacion": "Sí" if config["usar_estacion"] else "No",
        "Validacion": config["validacion"],
        "N_folds": len(resultados),
        "N_features": len(features),
        "Hiperparametros": str(config["xgb_params"]),
        "RMSE": round(col("RMSE").mean(), 4),
        "MAE": round(col("MAE").mean(), 4),
        "R2": round(col("R2").mean(), 4),
        "MedAE": round(col("MedAE").mean(), 4),
        "RMSE_norm": round(col("RMSE_norm").mean(), 4),
        "R2_desv": round(col("R2").std(), 4),
        "Nota": config["nota"],
    }

    enc_fill = PatternFill("solid", start_color="375623")   # verde XGB
    enc_font = Font(bold=True, color="FFFFFF", name="Arial")

    if not os.path.exists(ruta):
        wb = Workbook()
        ws = wb.active
        ws.title = "Ejecuciones"
        ws.append(list(fila.keys()))
        for c in ws[1]:
            c.fill = enc_fill; c.font = enc_font
            c.alignment = Alignment(horizontal="center")

        wb.create_sheet("Detalle_Folds")

        ws_imp = wb.create_sheet("Importancia")
        ws_imp.append(["Fecha_ejecucion", "Target", "Variable",
                       "Importancia_XGB", "Ranking"])
        for c in ws_imp[1]:
            c.fill = enc_fill; c.font = enc_font

        ws_shap = wb.create_sheet("SHAP")
        ws_shap.append(["Fecha_ejecucion", "Target", "Variable",
                        "SHAP_medio_abs", "Direccion", "Ranking"])
        for c in ws_shap[1]:
            c.fill = enc_fill; c.font = enc_font

        wb.save(ruta)

    wb = load_workbook(ruta)
    wb["Ejecuciones"].append(list(fila.values()))

    ws_det = wb["Detalle_Folds"]
    ws_det.delete_rows(1, ws_det.max_row)
    ws_det.append(["Fold", "n_train", "n_test", "RMSE", "MAE",
                   "R2", "MedAE", "RMSE_norm"])
    for c in ws_det[1]:
        c.fill = enc_fill; c.font = enc_font
    for r in resultados:
        ws_det.append([r["fold"], r["n_train"], r["n_test"],
                       round(r["RMSE"], 4), round(r["MAE"], 4),
                       round(r["R2"], 4), round(r["MedAE"], 4),
                       round(r["RMSE_norm"], 4)])

    ws_ej = wb["Ejecuciones"]
    for col_cells in ws_ej.columns:
        ancho = max(len(str(c.value)) for c in col_cells if c.value) + 2
        ws_ej.column_dimensions[col_cells[0].column_letter].width = min(ancho, 40)

    wb.save(ruta)
    print(f"\n[Excel] Ejecución registrada en: {ruta}")


# ============================================================================
# BLOQUE 7 — MAIN
# Orquesta el flujo completo: datos → folds → evaluación → consola → Excel.
# ============================================================================
def main():
    X, y, df, features = cargar_datos(CONFIG)
    folds = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, features)
    guardar_excel(CONFIG, resultados, features)


if __name__ == "__main__":
    main()

  XGBOOST — Total_Parasitos_ | estación: sí | validación: aleatoria
  Fold  n_test  RMSE      MAE       R²        MedAE     RMSE_norm   
--------------------------------------------------------------------------------------
  1     476     1.9029    1.2211    0.7498    0.6639    0.7229      
  2     476     2.2302    1.3555    0.6994    0.6654    0.8221      
  3     476     1.8941    1.2129    0.7196    0.6900    0.7471      
  4     475     1.9120    1.2495    0.6841    0.7205    0.8018      
  5     475     2.0520    1.3465    0.6400    0.7997    0.8465      
--------------------------------------------------------------------------------------
  MEDIA         1.9982    1.2771    0.6986    0.7079    0.7881      

[Excel] Ejecución registrada en: Registro_XGBoost.xlsx


In [18]:
"""
==============================================================================
 XGBOOST — Optimización de hiperparámetros con GridSearchCV
==============================================================================
 Búsqueda en grilla guiada por MAE (robusto al sesgo del target).
 La combinación ganadora se reevalúa reportando las CINCO métricas y se
 registra en el Excel acumulativo de XGBoost, hoja "Optimizacion".

 Requiere: pip install xgboost

 Validación interna: KFold aleatorio (coherente con el estudio).

 BLOQUES:
   1 Configuración  2 Carga/preparación  3 Grilla + GridSearchCV
   4 Reevaluación con 5 métricas  5 Reporte consola  6 Guardado Excel  7 Main
==============================================================================
"""

import os
from datetime import datetime
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             median_absolute_error, r2_score, make_scorer)
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# Datos, target, features, y la GRILLA de hiperparámetros de XGBoost.
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_XGBoost.xlsx",
    "target"       : "Total_Parasitos_",     # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", #"Velocidad_Corriente_m_s",
    ],
    "usar_estacion": True,
    "col_estacion" : "Estacion",
    "n_folds"      : 5,
    "random_state" : 42,

    # Grilla de XGBoost. Los hiperparámetros clave son distintos a los de RF:
    #   n_estimators     : número de árboles (rondas de boosting)
    #   max_depth        : profundidad de cada árbol (XGB usa árboles poco profundos)
    #   learning_rate    : cuánto aporta cada árbol (clave en boosting)
    #   subsample        : fracción de filas por árbol (regularización)
    #   colsample_bytree : fracción de columnas por árbol (regularización)
    # 3 x 3 x 3 x 2 x 2 = 108 combinaciones x 5 folds = 540 fits
    "grilla" : {
        "n_estimators"    : [100, 300, 500],
        "max_depth"       : [3, 5, 7, 9],
        "learning_rate"   : [0.01, 0.05, 0.1],
        "subsample"       : [0.8, 1.0, 0.5],
        "colsample_bytree": [0.8, 1.0, 0.5],
    },
    "nota"         : "GridSearch XGBoost guiado por MAE",
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — GRILLA + GridSearchCV (guiado por MAE)
# ============================================================================
def optimizar(config, X, y):
    cv = KFold(n_splits=config["n_folds"], shuffle=True,
               random_state=config["random_state"])

    scorer_mae = make_scorer(mean_absolute_error, greater_is_better=False)

    base = XGBRegressor(
        random_state=config["random_state"], n_jobs=-1,
        objective="reg:squarederror",
    )

    grid = GridSearchCV(
        estimator=base,
        param_grid=config["grilla"],
        scoring=scorer_mae,
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X, y)
    return grid, cv


# ============================================================================
# BLOQUE 4 — REEVALUACIÓN DEL MEJOR MODELO CON LAS 5 MÉTRICAS
# ============================================================================
def reevaluar(config, X, y, mejores_params, cv):
    modelo = XGBRegressor(
        random_state=config["random_state"], n_jobs=-1,
        objective="reg:squarederror", **mejores_params
    )
    pred = cross_val_predict(modelo, X, y, cv=cv, n_jobs=-1)

    rmse  = np.sqrt(mean_squared_error(y, pred))
    media = y.mean()
    return {
        "RMSE": rmse,
        "MAE": mean_absolute_error(y, pred),
        "R2": r2_score(y, pred),
        "MedAE": median_absolute_error(y, pred),
        "RMSE_norm": rmse / media if media != 0 else np.nan,
    }


# ============================================================================
# BLOQUE 5 — REPORTE POR CONSOLA
# ============================================================================
def reportar(config, grid, metricas, features):
    print("=" * 80)
    print("  XGBOOST — OPTIMIZACIÓN GridSearchCV (guía: MAE)")
    print("=" * 80)
    print(f"  Target        : {config['target']}")
    print(f"  Estación      : {'Sí' if config['usar_estacion'] else 'No'}")
    print(f"  Features      : {len(features)}")
    print(f"  Combinaciones : {len(grid.cv_results_['params'])}  "
          f"(x {config['n_folds']} folds)")
    print(f"\n  Mejores hiperparámetros encontrados:")
    for k, v in grid.best_params_.items():
        print(f"     {k:<20}: {v}")
    print(f"\n  MAE de la búsqueda (CV) : {-grid.best_score_:.4f}")
    print(f"\n  Métricas del mejor modelo (predicción out-of-fold):")
    print(f"     RMSE      : {metricas['RMSE']:.4f}")
    print(f"     MAE       : {metricas['MAE']:.4f}")
    print(f"     R²        : {metricas['R2']:.4f}")
    print(f"     MedAE     : {metricas['MedAE']:.4f}")
    print(f"     RMSE_norm : {metricas['RMSE_norm']:.4f}")
    print("=" * 80)


# ============================================================================
# BLOQUE 6 — GUARDADO EN EXCEL (hoja "Optimizacion", acumulativa)
# ============================================================================
def guardar_excel(config, grid, metricas, features):
    ruta = config["excel_salida"]
    enc_fill = PatternFill("solid", start_color="7030A0")
    enc_font = Font(bold=True, color="FFFFFF", name="Arial")

    cabecera = ["Fecha_ejecucion", "Modelo", "Target", "Estacion",
                "N_features", "Mejores_parametros", "MAE_CV",
                "RMSE", "MAE", "R2", "MedAE", "RMSE_norm", "Nota"]
    fila = [
        datetime.now().strftime("%Y-%m-%d %H:%M"), "XGBoost",
        config["target"], "Sí" if config["usar_estacion"] else "No",
        len(features), str(grid.best_params_), round(-grid.best_score_, 4),
        round(metricas["RMSE"], 4), round(metricas["MAE"], 4),
        round(metricas["R2"], 4), round(metricas["MedAE"], 4),
        round(metricas["RMSE_norm"], 4), config["nota"],
    ]

    if not os.path.exists(ruta):
        wb = Workbook(); wb.active.title = "Ejecuciones"; wb.save(ruta)

    wb = load_workbook(ruta)
    if "Optimizacion" not in wb.sheetnames:
        ws = wb.create_sheet("Optimizacion")
        ws.append(cabecera)
        for c in ws[1]:
            c.fill = enc_fill; c.font = enc_font
            c.alignment = Alignment(horizontal="center")
    ws = wb["Optimizacion"]
    ws.append(fila)

    for col_cells in ws.columns:
        ancho = max(len(str(c.value)) for c in col_cells if c.value) + 2
        ws.column_dimensions[col_cells[0].column_letter].width = min(ancho, 45)

    wb.save(ruta)
    print(f"\n[Excel] Optimización registrada en: {ruta} (hoja 'Optimizacion')")


# ============================================================================
# BLOQUE 7 — MAIN
# ============================================================================
def main():
    X, y, df, features = cargar_datos(CONFIG)
    grid, cv = optimizar(CONFIG, X, y)
    metricas = reevaluar(CONFIG, X, y, grid.best_params_, cv)
    reportar(CONFIG, grid, metricas, features)
    guardar_excel(CONFIG, grid, metricas, features)


if __name__ == "__main__":
    main()

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
  XGBOOST — OPTIMIZACIÓN GridSearchCV (guía: MAE)
  Target        : Total_Parasitos_
  Estación      : Sí
  Features      : 8
  Combinaciones : 324  (x 5 folds)

  Mejores hiperparámetros encontrados:
     colsample_bytree    : 1.0
     learning_rate       : 0.05
     max_depth           : 9
     n_estimators        : 500
     subsample           : 0.8

  MAE de la búsqueda (CV) : 1.0029

  Métricas del mejor modelo (predicción out-of-fold):
     RMSE      : 1.7133
     MAE       : 1.0030
     R²        : 0.7815
     MedAE     : 0.5049
     RMSE_norm : 0.6751

[Excel] Optimización registrada en: Registro_XGBoost.xlsx (hoja 'Optimizacion')


In [ ]:
"""
==============================================================================
 XGBOOST — SHAP por región + Perfiles de infestación  (OE3 + OE4)
==============================================================================
 Versión XGBoost, espejo del análisis de Random Forest, para comparar
 si ambos modelos coinciden en el ranking de variables (robustez del OE3)
 y en los perfiles regionales (OE4).

 Tres análisis integrados:
   A) SHAP GLOBAL          → ranking de importancia general (OE3)
   B) SHAP POR REGIÓN      → compara Aysén vs Magallanes (OE4)
   C) PERFILES INFESTACIÓN → rangos de variables físicas por nivel + estadios

 Requiere: pip install shap xgboost

 BLOQUES:
   1 Configuración   2 Carga/preparación   3 Entrenamiento
   4 SHAP global     5 SHAP por región     6 Perfiles de infestación
   7 Gráficos        8 Guardado Excel       9 Main
==============================================================================
"""

import os
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill
import shap
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_XGBoost.xlsx",
    "carpeta_img"  : "SHAP_XGB_region",
    "target"       : "Total_Parasitos_",   # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "estadios"     : ["Juveniles", "Adultos_Moviles", "Hembras_Ovigeras"],
    "col_region"   : "Region",
    "col_estacion" : "Estacion",
    "usar_estacion": True,
    "test_size"    : 0.25,
    "random_state" : 42,


    "hiperparametros": {
        ("Total_Parasitos_", False): {"colsample_bytree": 1.0, "learning_rate": 0.05,
                                      "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
        ("Total_Parasitos_", True) : {"colsample_bytree": 1.0, "learning_rate": 0.05,
                                      "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
        ("Hembras_Ovigeras", False): {"colsample_bytree": 1.0, "learning_rate": 0.05,
                                      "max_depth": 9, "n_estimators": 300, "subsample": 0.8},
        ("Hembras_Ovigeras", True) : {"colsample_bytree": 1.0, "learning_rate": 0.05,
                                      "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    region = df[config["col_region"]].copy()
    return X, y, region, df, features


# ============================================================================
# BLOQUE 3 — ENTRENAMIENTO (XGBoost con hiperparámetros óptimos)
# ============================================================================
def entrenar(config, X, y, region):
    clave = (config["target"], config["usar_estacion"])
    params = config["hiperparametros"][clave]
    print(f"  Hiperparámetros: {params}")
    idx = np.arange(len(X))
    Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
        X, y, idx, test_size=config["test_size"],
        random_state=config["random_state"]
    )
    modelo = XGBRegressor(random_state=config["random_state"], n_jobs=-1,
                          objective="reg:squarederror", **params)
    modelo.fit(Xtr, ytr)
    region_te = region.iloc[idx_te].values
    return modelo, Xte, yte, region_te


# ============================================================================
# BLOQUE 4 — SHAP GLOBAL
# ============================================================================
def shap_global(modelo, X_test, features):
    explainer = shap.TreeExplainer(modelo)
    shap_values = explainer.shap_values(X_test)
    importancia = np.abs(shap_values).mean(axis=0)
    direcciones = []
    for j in range(X_test.shape[1]):
        corr = np.corrcoef(X_test.iloc[:, j], shap_values[:, j])[0, 1]
        direcciones.append("positiva" if corr > 0 else "negativa")
    return explainer, shap_values, importancia, direcciones


# ============================================================================
# BLOQUE 5 — SHAP POR REGIÓN (OE4)
# ============================================================================
def shap_por_region(explainer, X_test, region_te, features):
    resultados = {}
    for reg in np.unique(region_te):
        mask = region_te == reg
        if mask.sum() < 10:
            continue
        sv = explainer.shap_values(X_test[mask])
        resultados[reg] = np.abs(sv).mean(axis=0)
    return resultados


# ============================================================================
# BLOQUE 6 — PERFILES DE INFESTACIÓN (bajo/medio/alto + estadios)
# ============================================================================
def perfiles_infestacion(config, X, y, region, df):
    p33 = y.quantile(0.33)
    p66 = y.quantile(0.66)

    def nivel(v):
        if v <= p33:   return "bajo"
        elif v <= p66: return "medio"
        else:          return "alto"
    niveles = y.apply(nivel)

    fisicas = config["features"]   

    def resumen(sub_idx, etiqueta):
        filas = []
        for nv in ["bajo", "medio", "alto"]:
            m = (niveles == nv) & sub_idx
            if m.sum() == 0:
                continue
            fila = {"Grupo": etiqueta, "Nivel": nv, "n": int(m.sum())}
            for f in fisicas:
                fila[f"{f}_p10"] = round(df.loc[m, f].quantile(0.10), 2)
                fila[f"{f}_med"] = round(df.loc[m, f].median(), 2)
                fila[f"{f}_p90"] = round(df.loc[m, f].quantile(0.90), 2)
            for est in config["estadios"]:
                fila[f"{est}_prom"] = round(df.loc[m, est].mean(), 2)
            filas.append(fila)
        return filas

    todo = pd.Series(True, index=y.index)
    perfil = resumen(todo, "GLOBAL")
    for reg in region.unique():
        perfil += resumen(region == reg, reg)
    return perfil, (p33, p66)


# ============================================================================
# BLOQUE 7 — GRÁFICOS
# ============================================================================
def graficos(config, shap_values, X_test, features, shap_reg):
    os.makedirs(config["carpeta_img"], exist_ok=True)
    suf = f"{config['target']}_{'conEst' if config['usar_estacion'] else 'sinEst'}"

    plt.figure()
    shap.summary_plot(shap_values, X_test, show=False)
    plt.tight_layout()
    plt.savefig(f"{config['carpeta_img']}/beeswarm_{suf}.png", dpi=150,
                bbox_inches="tight"); plt.close()

    if shap_reg:
        regiones = list(shap_reg.keys())
        x = np.arange(len(features))
        w = 0.8 / len(regiones)
        plt.figure(figsize=(9, 5))
        for i, reg in enumerate(regiones):
            plt.barh(x + i*w, shap_reg[reg], height=w, label=reg)
        plt.yticks(x + w*(len(regiones)-1)/2, features)
        plt.xlabel("SHAP medio absoluto")
        plt.title(f"SHAP por región (XGBoost) — {suf}")
        plt.legend(); plt.tight_layout()
        plt.savefig(f"{config['carpeta_img']}/region_{suf}.png", dpi=150,
                    bbox_inches="tight"); plt.close()
    print(f"  Gráficos en: {config['carpeta_img']}/")


# ============================================================================
# BLOQUE 8 — GUARDADO EN EXCEL
# ============================================================================
def guardar_excel(config, shap_reg, features, perfil, importancia, direcciones):
    ruta = config["excel_salida"]
    if not os.path.exists(ruta):
        wb = Workbook(); wb.active.title = "Ejecuciones"; wb.save(ruta)
    wb = load_workbook(ruta)
    fecha = datetime.now().strftime("%Y-%m-%d %H:%M")
    est_txt = "Sí" if config["usar_estacion"] else "No"
    fill = PatternFill("solid", start_color="375623")
    font = Font(bold=True, color="FFFFFF")

    # SHAP global
    if "SHAP" not in wb.sheetnames:
        ws = wb.create_sheet("SHAP")
        ws.append(["Fecha_ejecucion", "Target", "Estacion", "Variable",
                   "SHAP_medio_abs", "Direccion", "Ranking"])
        for c in ws[1]: c.fill = fill; c.font = font
    ws = wb["SHAP"]
    for rank, idx in enumerate(np.argsort(importancia)[::-1], 1):
        ws.append([fecha, config["target"], est_txt, features[idx],
                   round(float(importancia[idx]), 5), direcciones[idx], rank])

    # SHAP por región
    if "SHAP_Region" not in wb.sheetnames:
        ws = wb.create_sheet("SHAP_Region")
        ws.append(["Fecha", "Target", "Estacion", "Region", "Variable",
                   "SHAP_medio_abs", "Ranking"])
        for c in ws[1]: c.fill = fill; c.font = font
    ws = wb["SHAP_Region"]
    for reg, imp in shap_reg.items():
        for rank, idx in enumerate(np.argsort(imp)[::-1], 1):
            ws.append([fecha, config["target"], est_txt, reg, features[idx],
                       round(float(imp[idx]), 5), rank])

    # Perfiles
    if "Perfiles_Infestacion" not in wb.sheetnames:
        ws2 = wb.create_sheet("Perfiles_Infestacion")
        cab = ["Fecha", "Target", "Grupo", "Nivel", "n"]
        for f in config["features"]:
            cab += [f"{f}_p10", f"{f}_med", f"{f}_p90"]
        for est in config["estadios"]:
            cab += [f"{est}_prom"]
        ws2.append(cab)
        for c in ws2[1]: c.fill = fill; c.font = font
    ws2 = wb["Perfiles_Infestacion"]
    for fila in perfil:
        row = [fecha, config["target"], fila["Grupo"], fila["Nivel"], fila["n"]]
        for f in config["features"]:
            row += [fila[f"{f}_p10"], fila[f"{f}_med"], fila[f"{f}_p90"]]
        for est in config["estadios"]:
            row += [fila[f"{est}_prom"]]
        ws2.append(row)

    wb.save(ruta)
    print(f"  Registrado en Excel: hojas SHAP, SHAP_Region y Perfiles_Infestacion")


# ============================================================================
# BLOQUE 9 — MAIN
# ============================================================================
def main():
    print("=" * 70)
    print(f"  SHAP REGIÓN + PERFILES — XGBOOST — {CONFIG['target']} — "
          f"estación: {'Sí' if CONFIG['usar_estacion'] else 'No'}")
    print("=" * 70)

    X, y, region, df, features = cargar_datos(CONFIG)
    modelo, X_test, y_test, region_te = entrenar(CONFIG, X, y, region)

    explainer, shap_values, importancia, direcciones = shap_global(
        modelo, X_test, features)
    print("\n  SHAP GLOBAL (ranking):")
    for rank, idx in enumerate(np.argsort(importancia)[::-1], 1):
        print(f"    {rank}. {features[idx]:<24} "
              f"{importancia[idx]:.4f} ({direcciones[idx]})")

    shap_reg = shap_por_region(explainer, X_test, region_te, features)
    print("\n  SHAP POR REGIÓN:")
    for reg, imp in shap_reg.items():
        print(f"    {reg}:")
        for rank, idx in enumerate(np.argsort(imp)[::-1], 1):
            print(f"        {rank}. {features[idx]:<22} {imp[idx]:.4f}")

    perfil, (p33, p66) = perfiles_infestacion(CONFIG, X, y, region, df)
    print(f"\n  PERFILES (bajo<={p33:.2f}, medio<={p66:.2f}, alto>{p66:.2f}) → ver Excel")

    graficos(CONFIG, shap_values, X_test, features, shap_reg)
    guardar_excel(CONFIG, shap_reg, features, perfil, importancia, direcciones)
    print("\n  Completado.")


if __name__ == "__main__":
    main()

  SHAP REGIÓN + PERFILES — XGBOOST — Total_Parasitos_ — estación: Sí
  Hiperparámetros: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 500, 'subsample': 0.8}

  SHAP GLOBAL (ranking):
    1. Temperatura_C            1.5239 (positiva)
    2. Est_Invierno             0.8432 (positiva)
    3. Salinidad_psu            0.7396 (positiva)
    4. Est_Primavera            0.5162 (positiva)
    5. Altura_Mar_m             0.2274 (positiva)
    6. Capa_Mezcla_m            0.2270 (positiva)
    7. Velocidad_Corriente_m_s  0.1467 (positiva)
    8. Est_Otoño                0.0805 (negativa)
    9. Est_Verano               0.0536 (negativa)

  SHAP POR REGIÓN:
    Aysen:
        1. Temperatura_C          1.2605
        2. Est_Invierno           0.9517
        3. Salinidad_psu          0.8507
        4. Est_Primavera          0.5680
        5. Altura_Mar_m           0.2695
        6. Capa_Mezcla_m          0.2598
        7. Velocidad_Corriente_m_s 0.1672
        8. Es

#Lectura de tiempo

In [ ]:
"""
==============================================================================
 XGBOOST — SHAP por región + Perfiles de infestación  (OE3 + OE4)
==============================================================================
 Versión XGBoost, espejo del análisis de Random Forest, para comparar
 si ambos modelos coinciden en el ranking de variables (robustez del OE3)
 y en los perfiles regionales (OE4).

 Tres análisis integrados:
   A) SHAP GLOBAL          → ranking de importancia general (OE3)
   B) SHAP POR REGIÓN      → compara Aysén vs Magallanes (OE4)
   C) PERFILES INFESTACIÓN → rangos de variables físicas por nivel + estadios

 Requiere: pip install shap xgboost

 BLOQUES:
   1 Configuración   2 Carga/preparación   3 Entrenamiento
   4 SHAP global     5 SHAP por región     6 Perfiles de infestación
   7 Gráficos        8 Guardado Excel       9 Main
==============================================================================
"""

import os
import time
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill
import shap
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_XGBoost.xlsx",
    "carpeta_img"  : "SHAP_XGB_region",
    "target"       : "Hembras_Ovigeras",   # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "estadios"     : ["Juveniles", "Adultos_Moviles", "Hembras_Ovigeras"],
    "col_region"   : "Region",
    "col_estacion" : "Estacion",
    "usar_estacion": True,
    "test_size"    : 0.25,
    "random_state" : 84,

    # Hiperparámetros óptimos de XGBoost por (target, estacion), de tu registro
    "hiperparametros": {
        ("Total_Parasitos_", False): {"colsample_bytree": 1.0, "learning_rate": 0.05,
                                      "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
        ("Total_Parasitos_", True) : {"colsample_bytree": 1.0, "learning_rate": 0.05,
                                      "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
        ("Hembras_Ovigeras", False): {"colsample_bytree": 1.0, "learning_rate": 0.05,
                                      "max_depth": 9, "n_estimators": 300, "subsample": 0.8},
        ("Hembras_Ovigeras", True) : {"colsample_bytree": 1.0, "learning_rate": 0.05,
                                      "max_depth": 7, "n_estimators": 500, "subsample": 0.8},
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    region = df[config["col_region"]].copy()
    return X, y, region, df, features


# ============================================================================
# BLOQUE 3 — ENTRENAMIENTO (XGBoost con hiperparámetros óptimos)
# ============================================================================
def entrenar(config, X, y, region):
    clave = (config["target"], config["usar_estacion"])
    params = config["hiperparametros"][clave]
    print(f"  Hiperparámetros: {params}")
    idx = np.arange(len(X))
    Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
        X, y, idx, test_size=config["test_size"],
        random_state=config["random_state"]
    )
    modelo = XGBRegressor(random_state=config["random_state"], n_jobs=-1,
                          objective="reg:squarederror", **params)
    # Medición del tiempo de entrenamiento (solo el .fit del modelo óptimo).
    # Se excluye la carga de datos, el cálculo de SHAP y el guardado en Excel,
    # para reflejar únicamente el costo de entrenamiento del modelo.
    t_inicio = time.perf_counter()
    modelo.fit(Xtr, ytr)
    tiempo_fit = time.perf_counter() - t_inicio
    print(f"  Tiempo de entrenamiento (.fit): {tiempo_fit:.4f} s")
    region_te = region.iloc[idx_te].values
    return modelo, Xte, yte, region_te, tiempo_fit


# ============================================================================
# BLOQUE 4 — SHAP GLOBAL
# ============================================================================
def shap_global(modelo, X_test, features):
    explainer = shap.TreeExplainer(modelo)
    shap_values = explainer.shap_values(X_test)
    importancia = np.abs(shap_values).mean(axis=0)
    direcciones = []
    for j in range(X_test.shape[1]):
        corr = np.corrcoef(X_test.iloc[:, j], shap_values[:, j])[0, 1]
        direcciones.append("positiva" if corr > 0 else "negativa")
    return explainer, shap_values, importancia, direcciones


# ============================================================================
# BLOQUE 5 — SHAP POR REGIÓN (OE4)
# ============================================================================
def shap_por_region(explainer, X_test, region_te, features):
    resultados = {}
    for reg in np.unique(region_te):
        mask = region_te == reg
        if mask.sum() < 10:
            continue
        sv = explainer.shap_values(X_test[mask])
        resultados[reg] = np.abs(sv).mean(axis=0)
    return resultados


# ============================================================================
# BLOQUE 6 — PERFILES DE INFESTACIÓN (bajo/medio/alto + estadios)
# Solo variables físicas continuas (nunca estación) + conteo de estadios.
# ============================================================================
def perfiles_infestacion(config, X, y, region, df):
    p33 = y.quantile(0.33)
    p66 = y.quantile(0.66)

    def nivel(v):
        if v <= p33:   return "bajo"
        elif v <= p66: return "medio"
        else:          return "alto"
    niveles = y.apply(nivel)

    fisicas = config["features"]   # solo físicas continuas

    def resumen(sub_idx, etiqueta):
        filas = []
        for nv in ["bajo", "medio", "alto"]:
            m = (niveles == nv) & sub_idx
            if m.sum() == 0:
                continue
            fila = {"Grupo": etiqueta, "Nivel": nv, "n": int(m.sum())}
            for f in fisicas:
                fila[f"{f}_p10"] = round(df.loc[m, f].quantile(0.10), 2)
                fila[f"{f}_med"] = round(df.loc[m, f].median(), 2)
                fila[f"{f}_p90"] = round(df.loc[m, f].quantile(0.90), 2)
            for est in config["estadios"]:
                fila[f"{est}_prom"] = round(df.loc[m, est].mean(), 2)
            filas.append(fila)
        return filas

    todo = pd.Series(True, index=y.index)
    perfil = resumen(todo, "GLOBAL")
    for reg in region.unique():
        perfil += resumen(region == reg, reg)
    return perfil, (p33, p66)


# ============================================================================
# BLOQUE 7 — GRÁFICOS
# ============================================================================
def graficos(config, shap_values, X_test, features, shap_reg):
    os.makedirs(config["carpeta_img"], exist_ok=True)
    suf = f"{config['target']}_{'conEst' if config['usar_estacion'] else 'sinEst'}"

    plt.figure()
    shap.summary_plot(shap_values, X_test, show=False)
    plt.tight_layout()
    plt.savefig(f"{config['carpeta_img']}/beeswarm_{suf}.png", dpi=150,
                bbox_inches="tight"); plt.close()

    if shap_reg:
        regiones = list(shap_reg.keys())
        x = np.arange(len(features))
        w = 0.8 / len(regiones)
        plt.figure(figsize=(9, 5))
        for i, reg in enumerate(regiones):
            plt.barh(x + i*w, shap_reg[reg], height=w, label=reg)
        plt.yticks(x + w*(len(regiones)-1)/2, features)
        plt.xlabel("SHAP medio absoluto")
        plt.title(f"SHAP por región (XGBoost) — {suf}")
        plt.legend(); plt.tight_layout()
        plt.savefig(f"{config['carpeta_img']}/region_{suf}.png", dpi=150,
                    bbox_inches="tight"); plt.close()
    print(f"  Gráficos en: {config['carpeta_img']}/")


# ============================================================================
# BLOQUE 8 — GUARDADO EN EXCEL
# ============================================================================
def guardar_excel(config, shap_reg, features, perfil, importancia, direcciones, tiempo_fit):
    ruta = config["excel_salida"]
    if not os.path.exists(ruta):
        wb = Workbook(); wb.active.title = "Ejecuciones"; wb.save(ruta)
    wb = load_workbook(ruta)
    fecha = datetime.now().strftime("%Y-%m-%d %H:%M")
    est_txt = "Sí" if config["usar_estacion"] else "No"
    fill = PatternFill("solid", start_color="375623")
    font = Font(bold=True, color="FFFFFF")

    # Hoja de tiempos de entrenamiento (solo el .fit del modelo óptimo)
    if "Tiempos" not in wb.sheetnames:
        wst = wb.create_sheet("Tiempos")
        wst.append(["Fecha", "Modelo", "Target", "Estacion",
                    "Tiempo_fit_seg"])
        for c in wst[1]: c.fill = fill; c.font = font
    wst = wb["Tiempos"]
    wst.append([fecha, "XGBoost", config["target"], est_txt,
                round(tiempo_fit, 4)])

    # SHAP global
    if "SHAP" not in wb.sheetnames:
        ws = wb.create_sheet("SHAP")
        ws.append(["Fecha_ejecucion", "Target", "Estacion", "Variable",
                   "SHAP_medio_abs", "Direccion", "Ranking"])
        for c in ws[1]: c.fill = fill; c.font = font
    ws = wb["SHAP"]
    for rank, idx in enumerate(np.argsort(importancia)[::-1], 1):
        ws.append([fecha, config["target"], est_txt, features[idx],
                   round(float(importancia[idx]), 5), direcciones[idx], rank])

    # SHAP por región
    if "SHAP_Region" not in wb.sheetnames:
        ws = wb.create_sheet("SHAP_Region")
        ws.append(["Fecha", "Target", "Estacion", "Region", "Variable",
                   "SHAP_medio_abs", "Ranking"])
        for c in ws[1]: c.fill = fill; c.font = font
    ws = wb["SHAP_Region"]
    for reg, imp in shap_reg.items():
        for rank, idx in enumerate(np.argsort(imp)[::-1], 1):
            ws.append([fecha, config["target"], est_txt, reg, features[idx],
                       round(float(imp[idx]), 5), rank])

    # Perfiles
    if "Perfiles_Infestacion" not in wb.sheetnames:
        ws2 = wb.create_sheet("Perfiles_Infestacion")
        cab = ["Fecha", "Target", "Grupo", "Nivel", "n"]
        for f in config["features"]:
            cab += [f"{f}_p10", f"{f}_med", f"{f}_p90"]
        for est in config["estadios"]:
            cab += [f"{est}_prom"]
        ws2.append(cab)
        for c in ws2[1]: c.fill = fill; c.font = font
    ws2 = wb["Perfiles_Infestacion"]
    for fila in perfil:
        row = [fecha, config["target"], fila["Grupo"], fila["Nivel"], fila["n"]]
        for f in config["features"]:
            row += [fila[f"{f}_p10"], fila[f"{f}_med"], fila[f"{f}_p90"]]
        for est in config["estadios"]:
            row += [fila[f"{est}_prom"]]
        ws2.append(row)

    wb.save(ruta)
    print(f"  Registrado en Excel: hojas SHAP, SHAP_Region y Perfiles_Infestacion")


# ============================================================================
# BLOQUE 9 — MAIN
# ============================================================================
def main():
    print("=" * 70)
    print(f"  SHAP REGIÓN + PERFILES — XGBOOST — {CONFIG['target']} — "
          f"estación: {'Sí' if CONFIG['usar_estacion'] else 'No'}")
    print("=" * 70)

    X, y, region, df, features = cargar_datos(CONFIG)
    modelo, X_test, y_test, region_te, tiempo_fit = entrenar(CONFIG, X, y, region)

    explainer, shap_values, importancia, direcciones = shap_global(
        modelo, X_test, features)
    print("\n  SHAP GLOBAL (ranking):")
    for rank, idx in enumerate(np.argsort(importancia)[::-1], 1):
        print(f"    {rank}. {features[idx]:<24} "
              f"{importancia[idx]:.4f} ({direcciones[idx]})")

    shap_reg = shap_por_region(explainer, X_test, region_te, features)
    print("\n  SHAP POR REGIÓN:")
    for reg, imp in shap_reg.items():
        print(f"    {reg}:")
        for rank, idx in enumerate(np.argsort(imp)[::-1], 1):
            print(f"        {rank}. {features[idx]:<22} {imp[idx]:.4f}")

    perfil, (p33, p66) = perfiles_infestacion(CONFIG, X, y, region, df)
    print(f"\n  PERFILES (bajo<={p33:.2f}, medio<={p66:.2f}, alto>{p66:.2f}) → ver Excel")

    graficos(CONFIG, shap_values, X_test, features, shap_reg)
    guardar_excel(CONFIG, shap_reg, features, perfil, importancia, direcciones, tiempo_fit)
    print("\n  Completado.")


if __name__ == "__main__":
    main()

  SHAP REGIÓN + PERFILES — XGBOOST — Hembras_Ovigeras — estación: Sí
  Hiperparámetros: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 500, 'subsample': 0.8}
  Tiempo de entrenamiento (.fit): 0.9510 s

  SHAP GLOBAL (ranking):
    1. Temperatura_C            0.5118 (positiva)
    2. Salinidad_psu            0.2945 (positiva)
    3. Est_Invierno             0.2556 (positiva)
    4. Est_Primavera            0.1469 (positiva)
    5. Altura_Mar_m             0.0981 (positiva)
    6. Capa_Mezcla_m            0.0739 (positiva)
    7. Est_Otoño                0.0366 (negativa)
    8. Est_Verano               0.0349 (negativa)

  SHAP POR REGIÓN:
    Aysen:
        1. Temperatura_C          0.4254
        2. Salinidad_psu          0.3127
        3. Est_Invierno           0.2835
        4. Est_Primavera          0.1639
        5. Altura_Mar_m           0.1108
        6. Capa_Mezcla_m          0.0820
        7. Est_Otoño              0.0423
        8. Est_Verano